# Тетрадь 1 - Данные: окружение, импорт, экспорт

Установка окружения, загрузка недельных данных S&P 500 (10 лет) и бенчмарка total-return, точечно-исторический состав, очистка и политика пропусков, экспорт снапшота и таблиц в `data/`.

> **Соглашение по структуре** (см. `SPEC.md` → «Ways of working»): тетрадь состоит из идейных блоков. Перед каждым блоком - текст «что делаем и почему, на чем основано». Внутри блока - код. После важных результатов - короткое пояснение. Тетрадь самодостаточна.

> **Статус:** пустой скелет. Наполняется поблочно после одобрения. Ниже - *предлагаемый* план блоков (TODO).

## Планируемые идейные блоки (TODO)

- [x] Блок 1. Установка окружения и зависимости
- [x] Блок 2. Источники данных и точечно-исторический состав (PIT)
- [ ] Блок 3. Загрузка недельных цен: S&P 500 + ^SP500TR
- [ ] Блок 4. Очистка, выравнивание, политика пропусков (логируем, не заполняем молча)
- [ ] Блок 5. Экспорт снапшота и таблиц в `data/snapshot` и `data/tables`
- [ ] Блок 6. Краткий EDA + выводы по качеству данных

## Блок 1. Установка окружения и зависимости

Прежде чем тянуть хоть один тикер, я фиксирую окружение: без этого воспроизводимость невозможна. Идея простая: вся тяжелая логика (загрузка данных, метрики, бэктест, оптимизация) живет в пакете `index_tracking` (папка `src/`), а тетрадь только вызывает ее и по ходу объясняет, что и зачем происходит. То есть тетрадь остается читаемой, а код - тестируемым и переиспользуемым.

Наши собственные функции я помечаю префиксом `custom_` (например, `custom_tracking_error`): увидел такой вызов - значит код наш, и для дебага надо провалиться в `src/index_tracking/`. Все, что без префикса - это библиотеки (`pandas`, `numpy`, `cvxpy` и так далее).

Отдельно про источник цен: привычный `yfinance` тут не работает. Он ходит через `curl_cffi` с подделкой браузерного TLS-отпечатка, а наш прокси перешифровывает трафик, а значит соединение рвется. Поэтому недельные цены я качаю своим тонким клиентом на обычном `requests` (детали - в блоке 3).

**Установка** (один раз в окружении):

```bash
pip install -e ".[dev,notebook]"
```

Ниже - только импорты и версии: фиксируем, на чем именно считаем.

In [1]:
# зафиксировать корень проекта: относительные пути (data/, results/) работают
# независимо от того, откуда запущена тетрадь
from index_tracking.paths import custom_chdir_to_project_root
custom_chdir_to_project_root()

import sys
import numpy as np
import pandas as pd
import scipy
import requests
import matplotlib
import cvxpy
import sklearn

import index_tracking as it

print("python        :", sys.version.split()[0])
for m in (np, pd, scipy, requests, matplotlib, cvxpy, sklearn):
    print(f"{m.__name__:<14}:", m.__version__)
print("index_tracking:", it.__version__)

python        : 3.11.15
numpy         : 2.4.6
pandas        : 3.0.3
scipy         : 1.17.1
requests      : 2.33.1
matplotlib    : 3.11.0
cvxpy         : 1.9.2
sklearn       : 1.9.0
index_tracking: 0.1.0


Версии зафиксированы, пакет `index_tracking` импортируется без ошибок. По итогу окружение готово: дальше переходим к источникам данных и точечно-историческому составу индекса.

## Блок 2. Источники данных: точечно-исторический состав (PIT)

Я реплицирую индекс целиком, а не отдельные акции, поэтому мне нужны не сегодняшние члены S&P 500, а все бумаги, которые хоть раз входили в него за мой период. Если взять только тех, кто в индексе сейчас, я выкину компании, которые обанкротились или вылетели из индекса, то есть получу survivorship bias и приукрашенный результат.

Поэтому нужен point-in-time состав: кто именно был в индексе на каждую конкретную дату. Источник я выбрал после сравнения (детали - в отчете): `fja05680/sp500`, файл «Historical Components & Changes (Updated)». Он дает на каждую дату изменения полный список тикеров с 1996 года, делистнутые имена внутри, и обновляется до 2026-06-30. То есть ровно то, что нужно.

Отдельно я делаю кросс-проверку по Wikipedia (текущий список плюс сектор и CIK): если свежий состав из `fja` совпадает с Wikipedia, значит источнику можно доверять. Тикеры сразу нормализую под Yahoo (например, `BRK.B` записываю как `BRK-B`), потому что дальше по ним качаются цены.

In [2]:
from index_tracking.data import constituents as ct
from index_tracking import config as cfg

membership = ct.custom_load_sp500_membership()
print("строк (дат изменений):", len(membership))
print("период:", membership["date"].min().date(), "->", membership["date"].max().date())

# пример: кто был в индексе на конкретную дату
sample = ct.custom_members_on(membership, "2018-06-01")
print("членов на 2018-06-01:", len(sample))

строк (дат изменений): 2718
период: 1996-01-02 -> 2026-06-30
членов на 2018-06-01: 506


На каждую дату изменения состава у меня есть полный список членов. `custom_members_on` отдает состав на любую дату (берет последнюю запись не позже нее), а значит на каждом ребалансе я беру ровно тех, кто реально был в индексе тогда, без заглядывания вперед.

In [3]:
# вселенная активов за наш горизонт (все, кто хоть раз входил)
universe = ct.custom_universe(membership, start=cfg.WINDOW_START, end=cfg.WINDOW_END)
print(f"вселенная за {cfg.WINDOW_START}..{cfg.WINDOW_END}: {len(universe)} тикеров")

# кросс-проверка свежего состава по Wikipedia
members_latest = ct.custom_members_on(membership, cfg.WINDOW_END)
wiki = ct.custom_load_wikipedia_sp500()
only_fja = members_latest - set(wiki['ticker'])
only_wiki = set(wiki['ticker']) - members_latest
print("текущих членов (fja):", len(members_latest), "| Wikipedia:", len(wiki))
print("расхождение: только fja =", len(only_fja), "| только wiki =", len(only_wiki))

вселенная за 2015-01-01..2026-06-30: 768 тикеров


текущих членов (fja): 503 | Wikipedia: 503
расхождение: только fja = 0 | только wiki = 0


Вселенная за период - 768 тикеров против 503 сейчас: то есть за 11 лет через индекс прошло заметно больше имен, и это ровно те делистнутые и выбывшие бумаги, которые survivorship-подход потерял бы. Сверка с Wikipedia дала ноль расхождений (503 = 503), так что составу доверяем.

In [4]:
import os
import pandas as pd
os.makedirs("data/tables", exist_ok=True)

# вселенная -> csv
pd.DataFrame({"ticker": universe}).to_csv("data/tables/universe.csv", index=False)

# состав по датам -> parquet (тикеры компактно, строкой через запятую)
export = membership.copy()
export["tickers"] = export["tickers"].map(lambda ts: ",".join(ts))
export.to_parquet("data/tables/constituents_membership.parquet", index=False)

# секторы/CIK из Wikipedia -> csv (обогащение для анализа)
wiki.to_csv("data/tables/wikipedia_current.csv", index=False)
print("экспортировано в data/tables/:", sorted(os.listdir("data/tables")))

экспортировано в data/tables/: ['.gitkeep', 'constituents_membership.parquet', 'universe.csv', 'wikipedia_current.csv']


**Вывод блока 2.** Состав готов и проверен: PIT-вселенная на 768 тикеров, кросс-чек с Wikipedia чистый, все выгружено в `data/tables/`. Дальше по этой вселенной качаем недельные цены и total-return бенчмарк (блок 3).